In [1]:
import os
from dotenv import load_dotenv
#from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chat_models import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver

# Load API keys
load_dotenv(".env")
#google_api_key = os.getenv("GOOGLE_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
#llm = ChatGoogleGenerativeAI(
   # model="gemini-2.5-flash",   # or "gemini-2.5-pro" for higher quality
   # temperature=0,
    #google_api_key=google_api_key,
#)

# Tool: Simple QA
qa_prompt = PromptTemplate.from_template("Answer clearly: {question}")

@tool
def simple_qa(question: str) -> str:
    """Answer factual questions clearly."""
    chain = qa_prompt | llm
    return chain.invoke({"question": question}).content

tools = [simple_qa]

# Agent (add checkpointer=MemorySaver() here if you need multi-turn memory)
agent_executor = create_agent(model=llm, tools=tools)

if __name__ == "__main__":
    result = agent_executor.invoke(
        {"messages": [{"role": "user", "content": "What is LangGraph in LangChain?"}]}
    )
    print(result["messages"][-1].content)

ImportError: cannot import name 'ChatOpenAI' from 'langchain.chat_models' (C:\Users\Manikandan\anaconda3\envs\langchain\Lib\site-packages\langchain\chat_models\__init__.py)

In [ ]:
#!pip install langchain-classic

In [ ]:
#!pip install langchain langchain-core langchain-community langchain-google-genai langgraph python-dotenv

In [ ]:
# 🧠 Memory with LangChain v1 (checkpointer, not ConversationBufferMemory)
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent

# Memory (stores chat history) — MemorySaver takes NO constructor args
memory = MemorySaver()

# Agent with tool + memory
agent_executor = create_agent(
    model=llm,          # assumes `llm` is already defined (ChatGoogleGenerativeAI)
    tools=[simple_qa],  # assumes `simple_qa` tool is already defined
    checkpointer=memory,
)

# Every call needs a thread_id so the checkpointer knows which conversation to save/load
config = {"configurable": {"thread_id": "conversation-1"}}

def ask(question: str):
    result = agent_executor.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
    )
    return result["messages"][-1].content

# Run multiple interactions
print("1️⃣ First Question")
res1 = ask("What is LangChain?")
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = ask("Who created it?")
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = ask("Explain it simply again.")
print("\nAnswer:", res3)

# View full conversation history stored by the checkpointer
state = agent_executor.get_state(config)
for msg in state.values["messages"]:
    role = getattr(msg, "type", getattr(msg, "role", "unknown")).upper()
    print(f"{role}: {msg.content}")

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver

# Load API keys
load_dotenv(".env")
google_api_key = os.getenv("GOOGLE_API_KEY")

# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   # or "gemini-2.5-pro" for higher quality
    temperature=0,
    google_api_key=google_api_key,
)

# Tool: Simple QA
qa_prompt = PromptTemplate.from_template("Answer clearly: {question}")

@tool
def simple_qa(question: str) -> str:
    """Answer factual questions clearly."""
    chain = qa_prompt | llm
    return chain.invoke({"question": question}).content

tools = [simple_qa]

# Agent (add checkpointer=MemorySaver() here if you need multi-turn memory)
agent_executor = create_agent(model=llm, tools=tools)

if __name__ == "__main__":
    result = agent_executor.invoke(
        {"messages": [{"role": "user", "content": "What is LangGraph in LangChain?"}]}
    )
    print(result["messages"][-1].content)

In [ ]:
# 3) Summarized memory with LangChain v1
# ConversationSummaryMemory -> SummarizationMiddleware (built-in, no custom code needed)
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

memory = MemorySaver()

# Summarizes older messages once the conversation gets long, instead of
# either keeping everything (ConversationBufferMemory) or hard-cutting to
# the last k (ConversationBufferWindowMemory).
summarization = SummarizationMiddleware(
    model=llm,
    trigger=("messages", 6),   # summarize once history exceeds 6 messages
    keep=("messages", 2),      # always keep the 2 most recent messages verbatim
)

agent_executor = create_agent(
    model=llm,
    tools=[simple_qa],       # assumes `simple_qa` tool is already defined
    checkpointer=memory,
    middleware=[summarization],
)

config = {"configurable": {"thread_id": "conversation-1"}}

def ask(question: str):
    result = agent_executor.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
    )
    return result["messages"][-1].content

print("1️⃣ First Question")
res1 = ask("What is LangChain?")
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = ask("Who created it?")
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = ask("Explain it simply again.")
print("\nAnswer:", res3)

# View stored history (includes any summary message the middleware inserted)
state = agent_executor.get_state(config)
for msg in state.values["messages"]:
    role = getattr(msg, "type", getattr(msg, "role", "unknown")).upper()
    print(f"{role}: {msg.content}")

In [ ]:
from langchain.memory import VectorStoreRetrieverMemory
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

embedding = OpenAIEmbeddings()
retriever = FAISS.from_texts(["initial memory"], embedding).as_retriever()

memory = VectorStoreRetrieverMemory(retriever=retriever,memory_key="chat_history")
# Agent with tool + memory
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory
)

# Run multiple interactions
print("1️⃣ First Question")
res1 = agent.run("What is LangChain?")
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = agent.run("Who created it?")
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = agent.run("Explain it simply again.")
print("\nAnswer:", res3)